In [31]:
# importing dependecies
import random                          # Used to randomly pick one caption per image
from torch.utils.data import Dataset, DataLoader   # Tools for handling datasets and batching
from torchvision import transforms     # Image preprocessing utilities
from datasets import load_dataset      # To load dataset from Hugging Face
from collections import Counter        # Helps count word frequencies
import torch

In [32]:
# Loading the dataset
dataset = load_dataset("jxie/flickr8k")
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

In [33]:
train_data.features

{'image': Image(mode=None, decode=True),
 'caption_0': Value('string'),
 'caption_1': Value('string'),
 'caption_2': Value('string'),
 'caption_3': Value('string'),
 'caption_4': Value('string')}

In [34]:
# image transform resizing and normalizing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])])

In [35]:
def build_vocab(dataset, freq_threshold=4):
    """
    Builds a vocabulary (word -> index mapping)
    Parameters:
    - dataset: training dataset
    - freq_threshold: minimum frequency for a word to be included
    Returns:
    - word2idx: dictionary mapping word -> integer
    - idx2word: reverse mapping
    """

    counter = Counter()   # counts frequency of each word
    # Loop over all samples in dataset
    for sample in dataset:
        captions = [
           sample["caption_0"],
           sample["caption_1"],
           sample["caption_2"],
           sample["caption_3"],
           sample["caption_4"]]
        for caption in captions:
            tokens = caption.lower().split()
            counter.update(tokens)
    # Special tokens
    word2idx = {
        "<pad>": 0,    # used for padding shorter sequences
        "<start>": 1,  # start of sentence
        "<end>": 2,    # end of sentence
        "<unk>": 3  }   # unknown words
    idx = 4   # next index after special tokens

    # Add words that appear more than freq_threshold
    for word, freq in counter.items():
        if freq >= freq_threshold:
            word2idx[word] = idx
            idx += 1
    # Reverse mapping (index -> word)
    idx2word = {v: k for k, v in word2idx.items()}
    return word2idx, idx2word

In [36]:
# Build vocabulary only from training data
word2idx, idx2word = build_vocab(train_data)

In [37]:
def process_caption(caption, word2idx, max_len=20):
    """
    Converts a caption into:
    - input sequence
    - target sequence
    Steps:
    1. Tokenize
    2. Add <start> and <end>
    3. Convert words -> indices
    4. Pad to fixed length
    5. Create input-target pairs
    """
    tokens = caption.lower().split()   # Convert sentence to lowercase and split into words
    tokens = ["<start>"] + tokens + ["<end>"]  # Add special tokens
    # Convert words to indices and in case not found add <unk> special token
    sequence = [word2idx.get(word, word2idx["<unk>"]) for word in tokens]
    # Padding / truncation
    if len(sequence) < max_len:
        # Add <pad> tokens to reach max_len
        sequence += [word2idx["<pad>"]] * (max_len - len(sequence))
    else: sequence = sequence[:max_len] # truncate if the sequence is too long

    # Create input and target sequences
    input_seq  = sequence[:-1]
    target_seq = sequence[1:]
    return torch.tensor(input_seq), torch.tensor(target_seq)

In [38]:
class FlickrDataset(Dataset):
    """
    This class defines:
    - how many samples we have (__len__)
    - how to get one sample (__getitem__)
    """

    def __init__(self, data, word2idx, transform, max_len=20):
        """
        Parameters:
        - data: dataset split (train/val/test)
        - word2idx: vocabulary dictionary
        - transform: image preprocessing
        - max_len: max caption length
        """
        self.data = data
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        """Returns total number of samples"""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Returns ONE sample:
        - image tensor
        - input sequence
        - target sequence
        """
        sample = self.data[idx]
        image = sample["image"]          # PIL image
        captions = [                     # list of 5 captions
           sample["caption_0"],
           sample["caption_1"],
           sample["caption_2"],
           sample["caption_3"],
           sample["caption_4"]]
        caption = random.choice(captions) #randomly choose one

        # Apply preprocessing to image
        image = self.transform(image)
        # Convert caption to sequences
        input_seq, target_seq = process_caption(
            caption, self.word2idx, self.max_len)
        return image, input_seq, target_seq

In [39]:
# Create dataset instances for each split
train_dataset = FlickrDataset(train_data, word2idx, transform)
val_dataset   = FlickrDataset(val_data, word2idx, transform)
test_dataset  = FlickrDataset(test_data, word2idx, transform)

In [40]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [41]:
# Take one batch and print shapes
for images, inputs, targets in train_loader:
    print("Images shape :", images.shape)    # (batch_size, 3, 224, 224)
    print("Inputs shape :", inputs.shape)    # (batch_size, max_len-1)
    print("Targets shape:", targets.shape)   # (batch_size, max_len-1)
    break

Images shape : torch.Size([32, 3, 224, 224])
Inputs shape : torch.Size([32, 19])
Targets shape: torch.Size([32, 19])


In [42]:
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        # 1. Load pre-trained ResNet50 
        resnet = models.resnet50(weights='DEFAULT')
        
        # 2. Freeze all pre-trained layers 
        for param in resnet.parameters():
            param.requires_grad = False
            
        # 3. Remove the last classification layer (fc)
        # ResNet50's last layer before fc outputs a (1x1x2048) vector 
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        
        # 4. Add a fully connected layer to map 2048 to your embedding size 
        self.fc = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        # Extract features from ResNet
        with torch.no_grad():
            features = self.resnet(images) # Output: (batch_size, 2048, 1, 1)
        
        # Flatten for the linear layer
        features = features.view(features.size(0), -1) # Output: (batch_size, 2048)
        
        # Map to embedding space 
        features = self.bn(self.fc(features))
        return features

---
## Step 3 – Decoder: LSTM


In [ ]:
import torch
import torch.nn as nn

class DecoderLSTM(nn.Module):
    """
    LSTM caption decoder with teacher forcing.

    Input : image features (batch, embed_size)
             caption tokens  (batch, MAX_LEN)
    Output: logits           (batch, MAX_LEN-1, vocab_size)
    """
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1, dropout=0.5):
        super().__init__()
        self.hidden_size = hidden_size

        # Note: word2idx should be defined globally from your previous cells
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=word2idx["<pad>"])
        
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
       
        self.fc      = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

        self.init_h = nn.Linear(embed_size, hidden_size)
        self.init_c = nn.Linear(embed_size, hidden_size)

    def forward(self, features, captions):
    # Initialise LSTM hidden & cell state from image features
        h0 = self.init_h(features).unsqueeze(0)   
        c0 = self.init_c(features).unsqueeze(0)


        embeddings = self.dropout(self.embedding(captions))  


        lstm_out, _ = self.lstm(embeddings, (h0, c0))   

        logits = self.fc(self.dropout(lstm_out))         
        return logits

    def generate(self, feature, max_len=35):
        """
        Greedy decoding at inference time (no teacher forcing).
        feature : (1, embed_size)
        returns : list of word strings
        """
        self.eval()
        h = self.init_h(feature).unsqueeze(0)   # (1, 1, hidden_size)
        c = self.init_c(feature).unsqueeze(0)

        token  = torch.tensor([[word2idx["<start>"]]], device=feature.device)
        words  = []

        for _ in range(max_len):
            emb = self.embedding(token)                   # (1, 1, embed_size)
            out, (h, c) = self.lstm(emb, (h, c))          # (1, 1, hidden_size)
            logit = self.fc(out.squeeze(1))               # (1, vocab_size)
            pred  = logit.argmax(dim=1).item()

            if pred == word2idx["<end>"]:
                break
            if pred in (word2idx["<pad>"], word2idx["<unk>"]):
                continue

            words.append(idx2word.get(pred, '<unk>'))
            token = torch.tensor([[pred]], device=feature.device)

        return words


# --- Quick Test Block ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBED_SIZE = 256
HIDDEN_SIZE = 512
VOCAB_SIZE = len(word2idx) # Pulls dynamically from your built vocabulary
MAX_LEN = 20

dec_test = DecoderLSTM(EMBED_SIZE, HIDDEN_SIZE, VOCAB_SIZE).to(DEVICE)
with torch.no_grad():
    f = torch.randn(4, EMBED_SIZE).to(DEVICE)
    c = torch.randint(0, VOCAB_SIZE, (4, MAX_LEN)).to(DEVICE)
    o = dec_test(f, c)
print(f'Decoder output shape: {o.shape}  (expected [4, {MAX_LEN-1}, {VOCAB_SIZE}])')

Decoder output shape: torch.Size([4, 20, 2933])  (expected [4, 19, 2933])


In [44]:
#not complete yet
class ImageCaptioningModel(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers, dropout=0.5):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = EncoderCNN(embed_size)
        self.decoder = DecoderLSTM(embed_size, hidden_size, vocab_size, num_layers, dropout)

    def forward(self, images, captions):
        # Step 2: Extract features
        features = self.encoder(images)
        
        # Step 3: Pass features and captions to LSTM
        outputs = self.decoder(features, captions)
        return outputs

---
## Step 4 – Trianing


In [50]:
import torch.optim as optim

# Hyperparameters 
EMBED_SIZE = 256
HIDDEN_SIZE = 512
VOCAB_SIZE = len(word2idx)
NUM_LAYERS = 1
dropout = 0.75
LEARNING_RATE = 3e-4

model = ImageCaptioningModel(EMBED_SIZE, HIDDEN_SIZE, VOCAB_SIZE, NUM_LAYERS, dropout).to(DEVICE)

#  Cross Entropy Loss
criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<pad>"])
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [51]:
import torch.optim as optim
from tqdm import tqdm 

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30):
    # Optimal: Reduce learning rate when validation loss stops improving
    # settle the model into the best minimum
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        total_train_loss = 0
        
        for images, inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, inputs, targets = images.to(DEVICE), inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, inputs)
            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.reshape(-1))
            
            loss.backward()
            # Gradient clipping: prevents "Exploding Gradients" 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # --- VALIDATION PHASE ---
       
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for images, inputs, targets in val_loader:
                images, inputs, targets = images.to(DEVICE), inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(images, inputs)
                loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.reshape(-1))
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        
        # Step the scheduler based on val_loss
        scheduler.step(avg_val_loss)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        # Save the best model weights 
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print("--> Best Model Saved!")

# Run the optimized training
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30)

Epoch 1/30 [Train]: 100%|██████████| 188/188 [00:40<00:00,  4.61it/s]


Epoch [1/30] | Train Loss: 5.4824 | Val Loss: 4.3440
--> Best Model Saved!


Epoch 2/30 [Train]: 100%|██████████| 188/188 [00:42<00:00,  4.47it/s]


Epoch [2/30] | Train Loss: 4.4047 | Val Loss: 4.0486
--> Best Model Saved!


Epoch 3/30 [Train]: 100%|██████████| 188/188 [00:39<00:00,  4.74it/s]


...

Epoch [28/30] | Train Loss: 3.1999 | Val Loss: 3.0325
--> Best Model Saved!


Epoch 29/30 [Train]: 100%|██████████| 188/188 [00:36<00:00,  5.16it/s]


Epoch [29/30] | Train Loss: 3.1703 | Val Loss: 3.0019
--> Best Model Saved!


Epoch 30/30 [Train]: 100%|██████████| 188/188 [00:45<00:00,  4.16it/s]


Epoch [30/30] | Train Loss: 3.1493 | Val Loss: 2.9884
--> Best Model Saved!
